# 🎯 AI Recruiter Copilot
### LangChain · Groq Llama 3 70B · FAISS · SQLite · Gradio

**Objective:** Reuse existing candidates before sourcing externally.

---
**Run cells top-to-bottom on first launch. Subsequent runs can skip to Step 6.**

## Step 1 – Install Dependencies

In [ ]:
!pip install -q langchain langchain-groq langchain-community langchain-huggingface \
    faiss-cpu sentence-transformers gradio faker python-dotenv sqlalchemy \
    pandas numpy reportlab pypdf SpeechRecognition transformers torch plotly openai tiktoken

## Step 2 – Set API Key

In [ ]:
import os

# Option A: set directly
os.environ['GROQ_API_KEY'] = 'gsk_your_groq_api_key_here'

# Option B: load from .env
# from dotenv import load_dotenv
# load_dotenv()

os.environ.setdefault('DB_PATH', 'data/recruiter.db')
os.environ.setdefault('FAISS_INDEX_PATH', 'faiss_index')
os.environ.setdefault('EXPORT_PATH', 'exports')
os.environ.setdefault('GROQ_MODEL', 'llama3-70b-8192')
os.environ.setdefault('EMBEDDING_MODEL', 'sentence-transformers/all-MiniLM-L6-v2')

print('✅ Environment configured')

## Step 3 – Initialize Database & Seed 500 Candidates

In [ ]:
from modules.database import init_db
from modules.data_generator import seed_database

init_db()
seed_database(500)  # Skips if already seeded

In [ ]:
# Preview a sample candidate
import json
from modules.database import get_all_candidates

candidates = get_all_candidates()
sample = candidates[0]
print(f'Total candidates: {len(candidates)}')
print(f'\nSample Candidate:')
print(f'  Name: {sample["name"]}')
print(f'  Location: {sample["location"]}')
print(f'  Experience: {sample["experience_years"]} years')
print(f'  Skills: {sample["skills"]}')
print(f'  Industry: {sample["industry"]}')
print(f'\nAbout: {sample["about_section"][:200]}...')
print(f'\nProjects: {len(sample["projects"])} projects')
print(f'  First project: {sample["projects"][0]["title"]}')

## Step 4 – Build FAISS Index

In [ ]:
from modules.faiss_builder import build_faiss_index, index_exists, load_faiss_index
from modules.database import get_all_candidates

if index_exists():
    print('FAISS index already exists. Loading...')
    load_faiss_index()
else:
    print('Building FAISS index (this takes ~2-3 minutes)...')
    all_candidates = get_all_candidates()
    build_faiss_index(all_candidates)

print('✅ FAISS index ready')

## Step 5 – Demo: JD Intelligence Engine

In [ ]:
from modules.jd_intelligence import extract_jd, format_jd_display

sample_jd = """
We are hiring a Product Analyst to join our growing SaaS team in Bangalore.

Requirements:
- 2-4 years of experience in product analytics or business analytics
- Strong SQL skills for data extraction and analysis
- Experience with dashboarding tools (Tableau, Power BI, or similar)
- Excellent stakeholder management and communication skills
- Experience with A/B testing and experimentation frameworks
- Knowledge of product metrics (DAU, MAU, conversion funnels, retention)

Nice to have:
- Python for data analysis
- Experience with Mixpanel, Amplitude, or similar analytics tools
- Leadership experience managing junior analysts

Compensation: 12-20 LPA
Location: Bangalore (Hybrid)
"""

structured_jd = extract_jd(sample_jd)
print('Extracted JD Structure:')
print(json.dumps(structured_jd, indent=2))

## Step 6 – Demo: Hybrid Search (SQL + Semantic)

In [ ]:
import uuid
import pandas as pd
from modules.hybrid_search import run_hybrid_search

structured_jd['job_id'] = f'JOB-{str(uuid.uuid4())[:6].upper()}'
structured_jd['_raw_jd'] = sample_jd

job_id, ranked_candidates = run_hybrid_search(structured_jd, top_k=10)

print(f'\nJob ID: {job_id}')
print(f'Top {len(ranked_candidates)} Candidates Found:\n')

rows = []
for rank, c in enumerate(ranked_candidates, 1):
    rows.append({
        'Rank': rank,
        'Name': c['name'],
        'Location': c['location'],
        'Exp': c['experience_years'],
        'Match%': f"{c['match_score']:.1f}",
        'Confidence%': f"{c['confidence_score']:.1f}",
        'Engagement%': f"{c['engagement_score']:.1f}",
        'Final%': f"{c['final_score']:.1f}",
    })

df = pd.DataFrame(rows)
df

## Step 7 – Demo: Match Details (RAG)

In [ ]:
from modules.rag_engine import get_match_details
from modules.database import get_candidate

top_candidate = ranked_candidates[0]
cid = top_candidate['candidate_id']
candidate = get_candidate(cid)

print(f'Analyzing: {candidate["name"]} ({cid})')
print(f'Current Score: {top_candidate["final_score"]:.1f}%\n')

details = get_match_details(cid, structured_jd, candidate)

gap = details.get('gap_analysis', {})
print('MATCHED REQUIREMENTS:')
for m in gap.get('matched', []):
    print(f'  ✅ {m["requirement"]} — {m.get("strength", "")} [{m.get("evidence", "")}]')

print('\nMISSING REQUIREMENTS:')
for m in gap.get('missing', []):
    print(f'  ❌ {m["requirement"]} — Impact: {m.get("impact", "")}')

print(f'\nOverall: {gap.get("overall_gap_summary", "")}')
print(f'Match: {gap.get("match_percentage", 0)}%')

## Step 8 – Demo: AI Question Generation

In [ ]:
from modules.question_generator import generate_questions

gap = details.get('gap_analysis', {})
missing_skills = [m['requirement'] for m in gap.get('missing', [])][:3]

if not missing_skills:
    missing_skills = ['Leadership experience', 'Team management']

print(f'Generating questions for gaps: {missing_skills}\n')

context = candidate.get('about_section', '') + ' '.join(
    p.get('description', '') for p in candidate.get('projects', [])[:2]
)

questions = generate_questions(
    missing_skills,
    structured_jd.get('role', 'Product Analyst'),
    candidate['name'],
    context
)

print('Generated Screening Questions:')
for i, q in enumerate(questions, 1):
    print(f'\nQ{i}: {q["question"]}')
    print(f'   Targets: {q.get("targets_requirement", "")}')
    print(f'   Score Impact: {q.get("potential_score_impact", "")}')

## Step 9 – Demo: Candidate Assessment & Dynamic Re-Ranking

In [ ]:
from modules.question_generator import assess_answer
from modules.reranker import apply_assessment_and_rerank, get_reranked_candidates

# Simulate a candidate answer
question = questions[0]['question'] if questions else 'Have you managed a team?'
answer = """
Yes, I have managed a team of 4 junior analysts for the past 18 months at my current role.
I ran weekly sprint planning, conducted performance reviews, and mentored them on SQL and 
data visualization skills. I also worked directly with product managers and stakeholders 
to define analytics requirements and present findings to leadership.
"""

print(f'Question: {question}')
print(f'Answer: {answer[:100]}...')

result = assess_answer(
    question=question,
    answer=answer,
    role=structured_jd.get('role', 'Product Analyst'),
    requirement=questions[0].get('targets_requirement', 'Leadership') if questions else 'Leadership'
)

print(f'\nAssessment Results:')
print(f'  Verdict: {result["verdict"]}')
print(f'  Score: {result["assessment_score"]}/100')
print(f'  Score Impact: +{result["score_impact"]} points')
print(f'  Feedback: {result["feedback"]}')

# Apply to rankings
new_score = apply_assessment_and_rerank(
    job_id=job_id,
    candidate_id=cid,
    question=question,
    answer=answer,
    assessment_result=result,
    current_final_score=top_candidate['final_score']
)

print(f'\nDynamic Re-Ranking:')
print(f'  Before: {top_candidate["final_score"]:.1f}%')
print(f'  After:  {new_score:.1f}%')

## Step 10 – Demo: Workflow Tracker

In [ ]:
from modules.workflow import move_to_status, get_pipeline_summary
from modules.database import save_recruiter_memory

# Move top candidate through pipeline
move_to_status(job_id, cid, 'Contacted', 'Sent outreach email')
move_to_status(job_id, cid, 'Responded', 'Candidate confirmed interest')
move_to_status(job_id, cid, 'Screening Complete', 'Passed technical screening')

# Save recruiter memory
save_recruiter_memory(
    job_id, cid,
    question=question,
    answer=answer,
    notes='Strong leadership background. Ready for interview round.'
)

print('Pipeline Summary:')
summary = get_pipeline_summary(job_id)
for status, count in summary.items():
    if count > 0:
        print(f'  {status}: {count}')

## Step 11 – Demo: One-Click Submission Report

In [ ]:
from modules.submission import generate_submission_report, export_pdf, format_report_text

report = generate_submission_report(job_id, cid)
print(format_report_text(report))

In [ ]:
# Export PDF
pdf_path = export_pdf(report)
print(f'PDF saved: {pdf_path}')

## Step 12 – Demo: Feedback & Analytics

In [ ]:
from modules.feedback import record_outcome, get_analytics_chart, get_reutilization_rate

# Simulate feedback for a few candidates
record_outcome(job_id, cid, 'Interviewed', 'Strong technical skills')
record_outcome(job_id, ranked_candidates[1]['candidate_id'], 'Rejected', 'Not enough experience')
record_outcome(job_id, ranked_candidates[2]['candidate_id'], 'Hired', 'Best fit overall')

metrics = get_reutilization_rate()
print('Reutilization Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# Show analytics chart
fig = get_analytics_chart()
fig.show()

## Step 13 – Text-to-SQL Demo

In [ ]:
from modules.structured_search import structured_filter

# Direct structured search (fast path)
results = structured_filter(
    skills=['SQL', 'Dashboarding'],
    location='Bangalore',
    experience_min=2,
    experience_max=5,
    limit=10
)

print(f'Found {len(results)} Product Analysts in Bangalore with 2-5 years exp:')
for r in results[:5]:
    print(f'  {r["name"]} | {r["location"]} | {r["experience_years"]}yrs | {r["skills"][:3]}')

## Step 14 – Launch Gradio App

In [ ]:
# Launch the full Gradio UI
# The app will open at http://localhost:7860
import gradio_app
gradio_app.launch()